## Deep Dive into Multi-Turn Trajectory Mechanics

## 1. What Exactly Is a "Trajectory"?

In classical physics, a trajectory is the exact path an object takes through space over time (position, velocity at $t_0, t_1, t_2$).

In Reinforcement Learning, a trajectory (often denoted by the Greek letter $\tau$, "tau", or called a rollout / episode history) is the complete chronological recording of everything that happened from the moment the task started until it ended.

It is the unbroken log of:

- What the agent saw (Observation / State)
- What the agent decided to do (Action)
- What the external world answered with (Next Observation)
- What score was given (Reward)
- Whether the run was completed (Done / Termination flags)

Mathematically, a trajectory $\tau$ of length $H$ (horizon / number of turns) is represented as:

$$
\tau = \Big( s_0, a_0, r_0, s_1, a_1, r_1, s_2, a_2, r_2, \dots, s_H, a_H, r_H \Big)
$$

## 2. Single-Turn vs. Multi-Turn Trajectories

To understand why multi-turn agent trajectories are complex, contrast them directly:

### Single-Turn Trajectory

Single-Turn Trajectory (Module 1 & 2 - Math / Single Code Prompt):

Time $t=0$ ───> [Prompt] ───> [LLM generates entire solution] ───> [Verifier gives Reward] ───> FINISHED

(One shot, no back-and-forth feedback loop)

### Multi-Turn Trajectory

Multi-Turn Trajectory (Module 3 & 4 - Agent in OpenEnv):

Time $t=0$:  Env gives $Obs_0$ ("Bug in file X")
             └──> LLM emits $Act_0$ ("cat file X")

Time $t=1$:  Env runs command, returns $Obs_1$ ("Contents of file X...")
             └──> LLM reads $Obs_1$, emits $Act_1$ ("run pytest")

Time $t=2$:  Env runs pytest, returns $Obs_2$ ("FAILED: assert False == True")
             └──> LLM reads $Obs_2$, emits $Act_2$ ("fix file X & submit")

Time $t=3$:  Env runs tests, returns $Obs_3$ ("PASSED!"), Reward $= 1.0$, Done $= True$

In multi-turn, the agent does not generate the final answer in one breath. It generates intermediate commands, pauses, waits for the environment (bash/browser/database) to execute the command, inspects the environment's real output, and then decides what to do next.

## 3. How a Trajectory Is Represented in Memory & GPU Tensors

An LLM does not have separate physical ports for "eyes" (observations) and "hands" (actions). Everything is a single continuous sequence of token IDs inside its context window.

Let's look at how OpenEnv serializes a 2-turn trajectory into memory:

Token Index:    0 ... 49         50 ... 79          80 ... 280         281 ... 310

Content:       [Obs_0: Task]  [Act_0: "ls -la"]   [Obs_1: stdout]   [Act_1: "cat main.py"]

Source:        Environment     LLM Policy         Environment        LLM Policy

Who created:   OpenEnv Engine  Neural Net $\theta$       Linux Terminal     Neural Net $\theta$

When you pass this entire sequence into PyTorch for training, the input tensor $\text{input\_ids}$ looks like:

$$
\text{input\_ids} = [101, 4022, 912, \dots, 2045, 102]
$$

## 4. The Loss Mask (The Critical Sub-Component)

When training a neural network with backpropagation, the loss function looks at the model's logits and calculates cross-entropy or policy-gradient loss at every token position.

$$
\mathcal{L}(\theta) = -\sum_{i=1}^{N} M_i \cdot \log \pi_\theta(\text{token}_i \mid \text{token}_{<i}) \cdot A
$$

Look at that variable $M_i$. That is the Loss Mask (a binary vector of $0$s and $1$s of the exact same length as the token sequence).

Position i:      0   1   2 ... 49   50  51 ... 79   80  81 ... 280   281 282 ... 310

Token:          [--- Obs_0 ---]    [--- Act_0 ---]  [--- Obs_1 ---]  [--- Act_1 ---]

Mask ($M_i$):      0   0   0 ...  0    1   1 ...  1    0   0 ...   0     1   1 ...   1

Action Type:     Environment        LLM Policy       Environment       LLM Policy

Compute Grad?:   NO                 YES              NO                YES

Why is this masking absolutely mandatory?

The LLM is NOT an emulator of Linux: The 200 tokens in $Obs_1$ came from a Linux kernel running $ls -la$. The LLM's job is not to predict the timestamp, file size, or permissions that Linux outputs.

### Gradient Corruption

If you set $M_i = 1$ on the environment tokens, the optimizer forces the LLM's weights to increase the probability of generating the bash terminal's exact stdout text. This destroys the model's language representations because it wastes gradient capacity trying to memorize external terminal outputs.

### Credit Assignment

The policy gradient step should only reward or penalize decisions the policy actually made ($a_t$).

## 5. Answering the Edge Case

Now let's revisit what happens if your pipeline has a bug and sets the mask to $1$ for a 2,000-token compiler dump:

### Gradient Swamping

The 10 tokens of your actual action ($Act_1$) account for only $\frac{10}{2010} \approx 0.5\%$ of the gradient.

The other $99.5\%$ of the gradient update is spent trying to teach the LLM how to memorize compiler error strings.

The policy destabilizes, and training collapses.

## 1. Where Do the Non-Policy Tokens Come From?

During the live interaction loop, the LLM and the environment (OpenEnv / Linux sandbox) take turns writing to a shared text buffer.

Let's watch a real run step by step:

### Step 0: OpenEnv reset()

Text buffer starts with:

"Find the bug in main.py"  <-- (Environment token: Non-policy)

### Step 1: Forward Pass (Generation)

The LLM reads the buffer, samples tokens until `<EOS>`:

"cat main.py"  <-- (LLM Policy tokens)

Text buffer is now:

"Find the bug in main.py \n cat main.py"

### Step 2: Environment step("cat main.py")

OpenEnv runs the command in Linux, gets stdout, and appends it to the buffer:

"def add(a, b): return a - b"  <-- (Environment stdout: Non-policy)

Text buffer is now:

"Find the bug in main.py \n cat main.py \n def add(a, b): return a - b"

### Step 3: Forward Pass (Generation)

The LLM reads the entire buffer, samples tokens until `<EOS>`:

"fix line 1 to return a + b"  <-- (LLM Policy tokens)

At the end of the episode, the trajectory buffer contains a single continuous string of mixed text. Some parts were typed by the LLM, and some parts were dumped by the bash terminal / environment.

## 2. How the Trajectory Is Tokenized into Tensors

When the episode finishes, the tokenizer converts that entire string into a 1D tensor of token IDs:

$$
\text{input\_ids} = [t_0, t_1, t_2, t_3, t_4, t_5, t_6, t_7]
$$

Alongside this, your data pipeline constructs a Loss Mask tensor of the exact same length:

$$
\text{loss\_mask} = [0, 0, 1, 1, 0, 0, 1, 1]
$$

Token ID:    [ 101,   402,     912,    204,     881,    302,     105,    504 ]

Text:        [ "Find", "bug",  "cat", "main.py", "def", "return", "fix",  "add" ]

Source:      [ ---- Env ----]  [--- Policy ---]  [---- Env -----] [--- Policy --]

Loss Mask:   [   0,     0,       1,      1,       0,      0,       1,      1   ]

## 3. How Gradients Are Actually Computed (The Math & PyTorch Mechanism)

Now comes training time. We pass `input_ids` through the model in one single forward pass.

### Step A: Logits and Log-Probabilities

The Transformer computes the logits for every position in parallel. For each token $t_i$, we extract the model's log-probability of having predicted that token:

$$
\log \pi_\theta(\text{token}_i \mid \text{context}_{<i})
$$

This produces an array of log-probabilities of length $N$:

$$
\mathbf{L} = [l_0, l_1, l_2, l_3, l_4, l_5, l_6, l_7]
$$

### Step B: Element-Wise Mask Multiplication

We multiply the log-probabilities by our Loss Mask $\mathbf{M}$ and the trajectory's scalar Advantage $A$:

$$
\text{Token Loss}_i = M_i \cdot \log \pi_\theta(\text{token}_i \mid \text{context}_{<i}) \cdot A
$$

Look at what happens at each index:

Index $i$ | Source | Token | Log-Prob ($l_i$) | Mask ($M_i$) | Multiplied Loss Term
---|---|---|---|---|---
0 | Environment | "Find" | $-2.1$ | $0$ | $0 \times (-2.1) \times A = \mathbf{0.0}$
1 | Environment | "bug" | $-1.8$ | $0$ | $0 \times (-1.8) \times A = \mathbf{0.0}$
2 | LLM Policy | "cat" | $-0.4$ | $1$ | $1 \times (-0.4) \times A = \mathbf{-0.4A}$
3 | LLM Policy | "main.py" | $-0.2$ | $1$ | $1 \times (-0.2) \times A = \mathbf{-0.2A}$
4 | Environment | "def" | $-4.5$ | $0$ | $0 \times (-4.5) \times A = \mathbf{0.0}$
5 | Environment | "return" | $-3.9$ | $0$ | $0 \times (-3.9) \times A = \mathbf{0.0}$
6 | LLM Policy | "fix" | $-0.5$ | $1$ | $1 \times (-0.5) \times A = \mathbf{-0.5A}$
7 | LLM Policy | "add" | $-0.1$ | $1$ | $1 \times (-0.1) \times A = \mathbf{-0.1A}$

### Step C: Total Loss & Backpropagation

The total loss is the sum (or mean) over the active tokens:

$$
\mathcal{L}_{\text{total}}(\theta) = -\sum_{i} \left( M_i \cdot \log \pi_\theta(\text{token}_i) \cdot A \right) = -(-0.4A - 0.2A - 0.5A - 0.1A)
$$

When you call `loss.backward()`: PyTorch computes derivatives $\frac{\partial \mathcal{L}}{\partial \theta}$.

Because the terms for tokens $0, 1, 4, 5$ were multiplied by $0$, their gradients are strictly zero.

The network weights $\theta$ are updated exclusively based on the decisions made at tokens $2, 3, 6,$ and $7$.

## Inference vs. Training

Inference (generating the rollout step-by-step) is completely different from training (computing gradients over the recorded rollout).

They use two completely different execution modes of the Transformer. Let’s look at the exact mechanics of both, without skipping steps.

## 1. During Rollout / Environment Interaction (Inference Mode)

During the live interaction loop with OpenEnv, the model does do autoregressive generation step by step.

It does not get the whole future text at once because the future text does not exist yet.

Let's watch the exact loop:

### Step 0: Environment reset()

Buffer contains:

["Fix bug in math.py"]

### Step 1: Agent Turn 1 - Inference Generation

Pass the buffer to the LLM with `torch.no_grad()`.

The LLM generates autoregressively token by token:

- `cat`
- ` `
- `math.py`
- `<EOS>`

The generation stops immediately when the model emits `<EOS>`.

### Step 2: Environment Turn 1 - Tool Execution

OpenEnv intercepts `cat math.py`, runs it in Linux, and gets stdout:

`stdout = "def add(a, b): return a - b"`

Append stdout to the buffer.

Buffer is now:

["Fix bug in math.py", "cat math.py", "def add(a, b): return a - b"]

### Step 3: Agent Turn 2 - Inference Generation

Pass the updated buffer to the LLM with `torch.no_grad()`.

The LLM generates autoregressively token by token:

- `sed`
- ` -i`
- ` 's/-/+/'`
- ` math.py`
- `<EOS>`

Generation stops at `<EOS>`.

### Step 4: Environment Turn 2 - Tool Execution & Terminal Check

OpenEnv executes the command, runs tests, and sees that the tests pass.

The episode finishes. Reward $= +1.0$.

At this point, the episode is done. The entire recorded conversation is saved into memory as a completed trajectory log.

## 2. During Policy Gradient Update (Training / Forward Pass)

Now we enter the training step where we need to compute gradients and update model weights $\theta$.

Here is the key point: we do NOT generate any new text during the backward pass.

The trajectory is already recorded and fixed. We run a standard parallel teacher-forced forward pass over the recorded token sequence:

Recorded Input IDs: ["Fix", "bug", "cat", "math.py", "def", "add", "sed", "-i"]

Causal Mask: Standard Lower-Triangular Attention (Token $i$ only attends to $\le i$)

### One Single Forward Pass

We pass the entire recorded sequence into `model(input_ids)`. Because of the causal attention mask, the Transformer computes logits for all positions in one single parallel matrix multiplication on the GPU (no slow autoregressive loop).

### Extract Predicted Log-Probabilities

At every position $i$, the model outputs logits predicting token $i+1$. We compute the cross-entropy / log-probability of the actual token that was recorded in the trajectory:

$$
\log \pi_\theta(t_{i+1} \mid t_{\le i})
$$

### Apply the Loss Mask ($M$)

We zero out positions where the token came from the environment:

$$
\mathcal{L} = -\sum_{i} M_i \cdot \log \pi_\theta(t_{i+1} \mid t_{\le i}) \cdot A
$$

Position $i$:         0       1        2          3        4      5      6      7

Recorded Token:   "Fix"   "bug"    "cat"    "math.py"  "def"  "add"  "sed"  "-i"

Source:            Env     Env     Policy    Policy     Env    Env   Policy Policy

Loss Mask ($M_i$):    0       0        1          1        0      0      1      1

Advantage ($A$):     $+1.0$    $+1.0$     $+1.0$       $+1.0$     $+1.0$   $+1.0$   $+1.0$   $+1.0$

Loss Term:          0       0     $-\log P$     $-\log P$      0      0    $-\log P$ $-\log P$

Call `loss.backward()`: PyTorch backpropagates through the computational graph. Only the weights that contributed to predicting "cat", "math.py", "sed", and "-i" receive non-zero gradients.

## Why the Forward Pass Can Process All Tokens in Parallel

Let's look at the exact tensor mathematics of why a forward pass `model(input_ids)` processes all tokens in one single parallel matrix operation without looping.

## The Fundamental Difference

### Inference / Generation

You do not know what token comes next. You are forced to run the model in a loop:

generate token 1 $\to$ feed it back $\to$ generate token 2 $\to$ feed it back.

That is sequential and slow.

### Training / Scoring

The episode is already over. You already have the full text string recorded. You feed the entire sequence $[x_0, x_1, x_2, x_3]$ into the GPU simultaneously.

## How Does the GPU Compute the Logits for Every Step at Once?

The answer is the causal attention mask.

## Step-by-Step Mechanics: The Causal Attention Mask

Suppose our entire recorded trajectory has 4 tokens:

$$
\mathbf{X} = [\text{"Fix"}, \text{"bug"}, \text{"cat"}, \text{"math"}]
$$

When you call `logits = model(input_ids)`: 

### 1. Linear Projections (All in Parallel)

The embedding tensor $\mathbf{X} \in \mathbb{R}^{4 \times d}$ is multiplied by the weight matrices $W_Q, W_K, W_V$ in one single GEMM (General Matrix Multiply) on the GPU:

$$
Q = \mathbf{X} W_Q, \quad K = \mathbf{X} W_K, \quad V = \mathbf{X} W_V \quad (\text{Shapes: } 4 \times d)
$$

### 2. The Raw Attention Scores Matrix ($4 \times 4$)

The GPU computes the dot product of all Queries with all Keys at once:

$$
S = QK^T = \begin{bmatrix}
q_0 \cdot k_0 & q_0 \cdot k_1 & q_0 \cdot k_2 & q_0 \cdot k_3 \\
q_1 \cdot k_0 & q_1 \cdot k_1 & q_1 \cdot k_2 & q_1 \cdot k_3 \\
q_2 \cdot k_0 & q_2 \cdot k_1 & q_2 \cdot k_2 & q_2 \cdot k_3 \\
q_3 \cdot k_0 & q_3 \cdot k_1 & q_3 \cdot k_2 & q_3 \cdot k_3
\end{bmatrix}
$$

Notice: Row 0 represents what token 0 sees; Row 1 represents what token 1 sees; Row 2 represents what token 2 sees; Row 3 represents what token 3 sees.

### 3. Applying the Causal Mask (Lower-Triangular Mask)

To prevent token 0 from seeing tokens 1, 2, and 3, the Transformer applies a mask where the upper triangle is set to $-\infty$:

$$
\text{Masked } S = \begin{bmatrix}
q_0 \cdot k_0 & -\infty & -\infty & -\infty \\
q_1 \cdot k_0 & q_1 \cdot k_1 & -\infty & -\infty \\
q_2 \cdot k_0 & q_2 \cdot k_1 & q_2 \cdot k_2 & -\infty \\
q_3 \cdot k_0 & q_3 \cdot k_1 & q_3 \cdot k_2 & q_3 \cdot k_3
\end{bmatrix}
$$

### 4. The Softmax Step

When $\text{Softmax}(\cdot)$ is computed along each row, $e^{-\infty} = 0$:

$$
\text{Attention Weights} = \begin{bmatrix}
1.0 & 0 & 0 & 0 \\
w_{1,0} & w_{1,1} & 0 & 0 \\
w_{2,0} & w_{2,1} & w_{2,2} & 0 \\
w_{3,0} & w_{3,1} & w_{3,2} & w_{3,3}
\end{bmatrix}
$$

### What Does the Model Output at Each Position?

Because of the lower-triangular mask, each row in the final output tensor is computed strictly from the tokens that came before it, all at the same physical time on the GPU:

- Row 0: Sees only ["Fix"] → Produces logits predicting token 1 ("bug")
- Row 1: Sees only ["Fix", "bug"] → Produces logits predicting token 2 ("cat")
- Row 2: Sees only ["Fix", "bug", "cat"] → Produces logits predicting token 3 ("math")
- Row 3: Sees only ["Fix", "bug", "cat", "math"] → Produces logits predicting what would follow "math"

The output tensor `logits` has shape:

$$
[\text{Batch Size}, \text{Sequence Length}, \text{Vocab Size}] \to [1, 4, 32000]
$$

- `logits[0, 0, :]` are the unnormalized scores for the vocabulary given only the context ["Fix"].
- `logits[0, 1, :]` are the unnormalized scores for the vocabulary given only the context ["Fix", "bug"].
- `logits[0, 2, :]` are the unnormalized scores for the vocabulary given only the context ["Fix", "bug", "cat"].

## Contrast: Side-by-Side Execution Flow

### Inference / Generation (Sequential Loop)

Pass 1: Input: ["Fix"] → GPU computes 1 token → Emits "bug"

Pass 2: Input: ["Fix", "bug"] → GPU computes 1 token → Emits "cat"

Pass 3: Input: ["Fix", "bug", "cat"] → GPU computes 1 token → Emits "math"

Total: 3 separate GPU kernel launches (O(N) sequential steps).

### Training / Logits Evaluation (Single Parallel Pass)

Pass 1: Input: ["Fix", "bug", "cat", "math"]

→ GPU computes the entire $4 \times 4$ masked attention matrix at once.

→ Output: Tensor of shape $[1, 4, 32000]$ containing logits for all 4 steps simultaneously.

Total: 1 GPU kernel launch (O(1) step).

This parallel evaluation of the entire recorded sequence is called Teacher Forcing. It is the exact reason why calculating logits and losses over a 1000-token trajectory takes about 10 milliseconds during training, whereas generating that same 1000-token trajectory took several seconds during inference.

## Part 1: Why Is It Called Parallel / $O(1)$ Across Time If It Still Has $L$ Layers?

You are completely right: layers are sequential. Layer 2 cannot run until Layer 1 finishes, Layer 3 waits for Layer 2, and so on.

The key is distinguishing between two different axes:

- **Depth Axis (Layer Dimension)**: Sequential ($1 \to 2 \to \dots \to L$)
- **Time / Sequence Axis (Token Dimension)**: Parallel ($t_0, t_1, \dots, t_N$ processed simultaneously)

Let’s compare what the GPU actually does at Layer 1 in both modes.

### A. Inference (Sequential Along Time)

You have $N=4$ tokens.

1. Feed token 0 → Layer 1 → Layer 2 → ... → Layer $L$ → emit token 1
2. Feed token 1 → Layer 1 → Layer 2 → ... → Layer $L$ → emit token 2
3. Feed token 2 → Layer 1 → Layer 2 → ... → Layer $L$ → emit token 3
4. Feed token 3 → Layer 1 → Layer 2 → ... → Layer $L$ → emit token 4

Total forward passes through all $L$ layers = 4 passes ($N$ sequential calls).

### B. Training (Parallel Along Time)

You have the full recorded buffer $[t_0, t_1, t_2, t_3]$.

1. Feed all 4 tokens together into Layer 1.
2. Layer 1 computes attention across all 4 tokens simultaneously.
3. Layer 1 outputs a $(4, d)$ matrix. Feed it into Layer 2.
4. Layer 2 outputs a $(4, d)$ matrix. Feed it into Layer 3.
5. Continue through the remaining layers.
6. Layer $L$ outputs a $(4, d)$ matrix. Project to vocab → $(4, |V|)$.

Total forward passes through all $L$ layers = 1 pass (1 call).

When engineers say training is $O(1)$ in time steps / parallel, they mean all $N$ time steps travel through the $L$ layers together as a single batch matrix, rather than making $N$ separate sequential round-trips through the $L$ layers.

## Part 2: How Does PyTorch Actually Compute and Update Gradients Backwards?

Let’s trace the exact forward-to-backward tensor path for a 4-token trajectory:

$$
\text{Tokens: } [t_0, t_1, t_2, t_3] = [\text{"Fix"}, \text{"bug"}, \text{"cat"}, \text{"math"}]
$$

$$
\text{Sources: } [\text{Env}, \text{Env}, \text{Policy}, \text{Policy}]
$$

$$
\text{Loss Mask } M = [0, 0, 1, 1]
$$

$$
\text{Trajectory Advantage } A = +1.5
$$

### Step 1: The Forward Output Tensor

The forward pass through all $L$ layers outputs logits of shape $[4, |V|]$:

- Position 0 (Context: ["Fix"]): logits for predicting $t_1$ ("bug")
- Position 1 (Context: ["Fix", "bug"]): logits for predicting $t_2$ ("cat")
- Position 2 (Context: ["Fix", "bug", "cat"]): logits for predicting $t_3$ ("math")
- Position 3 (Context: All 4 tokens): logits for predicting $t_4$ (`<EOS>`)

### Step 2: Compute Log-Probabilities & the Scalar Loss

For each active position, we calculate the log-probability of the actual token that was chosen:

- For position 1 (predicting "cat"): $p_1 = \log \pi_\theta(\text{"cat"} \mid \text{"Fix"}, \text{"bug"})$
- For position 2 (predicting "math"): $p_2 = \log \pi_\theta(\text{"math"} \mid \text{"Fix"}, \text{"bug"}, \text{"cat"})$

We apply the loss mask $M$ and advantage $A$:

$$
\text{Loss} = -\left( \underbrace{0 \cdot p_0 \cdot A}_{\text{Env (0)}} + \underbrace{1 \cdot p_1 \cdot A}_{\text{Policy ("cat")}} + \underbrace{1 \cdot p_2 \cdot A}_{\text{Policy ("math")}} + \underbrace{0 \cdot p_3 \cdot A}_{\text{Env (0)}} \right)
$$

This reduces to a single scalar float:

$$
\text{Loss} = -(p_1 + p_2) \cdot 1.5
$$

### Step 3: The Backward Pass (`loss.backward()`)

PyTorch maintains a dynamic computational graph created during the forward pass. When `loss.backward()` is called, the chain rule flows backward from the scalar loss through all $L$ layers:

```text
Scalar Loss
    │
    ▼  dL / d(logits)
Final LM Head
    │
    ▼  dL / d(Layer L)
Layer L
    │
    ▼
    ...
    │
    ▼  dL / d(Layer 1)
Layer 1
    │
    ▼  dL / d(Weights W)
Accumulate Gradients:
   W.grad += dL / dW
```

For every trainable weight matrix $W$ in every layer:

$$
\nabla_W \text{Loss} = \frac{\partial \text{Loss}}{\partial p_1} \frac{\partial p_1}{\partial W} + \frac{\partial \text{Loss}}{\partial p_2} \frac{\partial p_2}{\partial W}
$$

Notice that $p_0$ and $p_3$ had a multiplier of $0$, so their branch in the computational graph receives zero incoming gradient: $\frac{\partial \text{Loss}}{\partial p_0} = 0$.

Only the activations that produced "cat" and "math" send back non-zero gradient signals.

### Step 4: The Optimizer Step (`optimizer.step()`)

Once backpropagation completes, every trainable weight tensor $W$ has its `.grad` field populated. The optimizer (for example, AdamW) updates the weights:

$$
W_{\text{new}} = W_{\text{old}} - \eta \cdot \text{Adam}(\nabla_W \text{Loss})
$$

Because $A = +1.5$ (positive advantage), the update shifts $W$ in the exact direction that increases the probability that the model will emit "cat" and "math" when presented with those contexts in the future.

## Summary Checklist

- **Parallel execution**: all sequence positions are processed through all $L$ layers in a single pass via batched matrix multiplications, constrained by the causal mask so earlier tokens cannot see later tokens.
- **Scalar loss reduction**: log-probabilities are multiplied element-wise by the binary mask $M$ and scalar advantage $A$, then summed into a single loss scalar.
- **Backpropagation**: the chain rule flows back through all $L$ layers, populating gradients strictly for the token decisions that had $M_i = 1$.